# Multi-speaker dialogue generation with FireRedTTS‑2 and OpenVINO

FireRedTTS‑2 is a long-form streaming TTS system for multi-speaker dialogue generation, delivering stable, natural speech with reliable speaker switching and context-aware prosody. It is highlighted by following features:
- **Long Conversational Speech Generation**: It currently supports 3 minutes dialogues with 4 speakers and can be easily scaled to longer conversations
with more speakers by extending training corpus.
- **Multilingual Support**: It supports multiple languages including English, Chinese, Japanese, Korean, French, German, and Russian. Support zero-shot voice cloning for cross-lingual and code-switching scenarios.
- **Ultra-Low Latency**: Building on the new **12.5Hz streaming** speech tokenizer, we employ a dual-transformer architecture that operates on a text–speech interleaved sequence, enabling flexible sentence-by-sentence generation and reducing first-packet latency，Specifically, on an L20 GPU, our first-packet latency as low as 140ms while maintaining high-quality audio output.
- **Strong Stability**：Our model achieves high similarity and low WER/CER in both monologue and dialogue tests.
- **Random Timbre Generation**:Useful for creating ASR/speech interaction data.

More details can be found in the [paper](https://arxiv.org/abs/2509.02020), original [repository](https://github.com/FireRedTeam/FireRedTTS2) and [model card](https://huggingface.co/FireRedTeam/FireRedTTS2)

In this tutorial we consider how to run and optimize FireRedTTS‑2 using OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and Optimize model](#Convert-and-Optimize-model)
- [Create Inference Pipeline](#Create-Inference-Pipeline)
    - [Select Inference Device](#Select-Inference-Device)
    - [Run Dialogue Generation](#Run-Dialogue-Generation)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/janus-multimodal-generation/janus-multimodal-generation.ipynb" />


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/fireredtts2/fireredtts2.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
# Fetch `notebook_utils` module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("fireredtts2.ipynb")

In [ ]:
from cmd_helper import clone_repo
from pip_helper import pip_install
import platform

!pip uninstall -y fireredtts2

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch==2.7.1",
    "torchvision==0.22.1",
    "torchaudio==2.7.1",
    "nncf",
    "openvino>=2025.3.0",
    "transformers==4.55",
    "gradio",
    "huggingface_hub",
)

repo_dir = Path("FireRedTTS2")
revision = "bfacbfb7bb88cade9c0b9ab2644ebd7f75c6989c"
clone_repo("https://github.com/openvino-dev-samples/FireRedTTS2.git", revision)

pip_install(
    "-q -e",
    str(repo_dir),
)

pip_install(
    "-q -r",
    str(repo_dir / "requirements.txt"),
)
if platform.system() == "Darwin":
    pip_install("numpy<2.0")

Found existing installation: fireredtts2 0.1
Uninstalling fireredtts2-0.1:
  Successfully uninstalled fireredtts2-0.1



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
  DEPRECATION: Legacy editable install of fireredtts2==0.1 from file:///home/ethan/intel/openvino_notebooks/notebooks/fireredtts2/FireRedTTS2 (setup.py develop) is deprecated. pip 25.3 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)

FireRedTTS2 is a PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). [OpenVINO model conversion API](https://docs.openvino.ai/2024/openvino-workflow/model-preparation.html#convert-a-model-with-python-convert-model) should be used for these purposes. `ov.convert_model` function accepts original PyTorch model instance and example input for tracing and returns `ov.Model` representing this model in OpenVINO framework. Converted model can be used for saving on disk using `ov.save_model` function or directly loading on device using `core.compile_model`.

The script `ov_fireredtts_helper.py` contains helper function for model conversion.

**Model Components:**

- `openvino_text_embeddings_model` - Converts text tokens into embedding vectors for the language model processing
- `openvino_audio_embeddings_model` - Converts audio codebook tokens into embedding vectors for audio processing
- `openvino_audio_decoder_model` - Decodes acoustic features into audio waveforms from the encoded representations
- `openvino_audio_upsampler_model` - Upsamples audio tokens through RVQ decoding to generate higher quality audio features
- `openvino_audio_encoder_model` - Encodes raw audio waveforms into compressed token representations
- `openvino_decoder_model` - Transformer-based decoder that generates subsequent audio codebook levels (codebooks 1-15) autoregressively
- `openvino_backbone_model` - Main transformer backbone that processes text and audio embeddings to generate the first level audio codebook (codebook 0) and contextual representations


In [ ]:
from ov_fireredtts_helper import convert_fireredtts2
from huggingface_hub import snapshot_download

pt_model_path = "pretrained_models"
snapshot_download(repo_id="FireRedTeam/FireRedTTS2", local_dir=Path(pt_model_path))

model_path = "FireRedTTS2-ov"
convert_fireredtts2(pt_model_path, model_path)

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

✅ pretrained_models model already converted. You can find results in FireRedTTS2-ov


PosixPath('FireRedTTS2-ov')

## Create Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

`OVFireRedTTS2` defined in `ov_fireredtts_helper.py` provides unified interface for running model inference. It accepts model directory and target device for inference.

### Select Inference Device
[back to top ⬆️](#Table-of-contents:)

In [5]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU"])

device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

`OVFireRedTTS2` class used for pre- and postprocessing steps in original FireRedTTS-2 model. Our model is also compatible with the same processor code and we can reuse it. 

ℹ️ **Limitation**
- Currently it can support `dialogue mode` only. 
- Codec model can be deployed to `CPU` only.

In [6]:
from ov_fireredtts_helper import OVFireRedTTS2

ov_model = OVFireRedTTS2(model_path, gen_type="dialogue", device=device.value, codec_device="CPU")

[INFO] OV model Loaded...
[INFO] Text Tokenizer Loaded...


### Run Dialogue Generation
[back to top ⬆️](#Table-of-contents:)

<img width="395" height="306" alt="image" src="https://github.com/user-attachments/assets/a5a43024-327a-408c-a1cd-59de9f25e501" />


FireRedTTS2 is a text-to-speech model using a dual-transformer architecture with interleaved text–speech input, enabling sentence-by-sentence generation and contextually coherent prosody.
As illustrated in Figure, each dialogue text is prefixed with a speaker tag (e.g., "[S1]") and concatenated with its corresponding speech tokens; these segments are then joined in temporal order to form sequences such as "[S1]<text><audio>[S2]<text><audio>[S3]<text><audio>..."

In [7]:
import torchaudio


text_list = [
    "[S1]It's alright, we'll take a breath and plan the next pass together.",
    "[S2]Yeah, thanks. We'll get it right this time.",
    "[S1]Let's review our signals tonight so we're in sync on the field tomorrow.",
]
prompt_wav_list = [
    "FireRedTTS2/examples/chat_prompt/en/S1.flac",
    "FireRedTTS2/examples/chat_prompt/en/S2.flac",
]

prompt_text_list = [
    "[S1]I think we should just talk about what happened and move on because there's going to be other jousts and Sir Saif isn't done yet. It's not, he's not, it's not done yet.",
    "[S2]You know, maybe sorry, maybe maybe I pushed, maybe I pushed too hard. I was really excited. I didn't mean to make you snap.",
]

all_audio = ov_model.generate_dialogue(
    text_list=text_list,
    prompt_wav_list=prompt_wav_list,
    prompt_text_list=prompt_text_list,
    temperature=0.9,
    topk=30,
)
torchaudio.save("chat_clone_ov.wav", all_audio, 24000)

100%|████████████████████████████████████████████████████████████████████████████| 3/3 [00:20<00:00,  6.78s/it]


In [8]:
import IPython

display(IPython.display.Audio("chat_clone_ov.wav"))

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/